# HunyuanVideo 1.5 — Colab T4

**Быстрая генерация видео из текста** с HunyuanVideo 1.5 (Tencent) на бесплатном T4 GPU.

### Почему HunyuanVideo?
| | HunyuanVideo 1.5 | Wan 2.2 14B |
|---|---|---|
| **Размер модели** | ~7.76 ГБ (FP8) | ~14 ГБ (FP8) |
| **Скорость** | ~2x быстрее | Базовая |
| **Качество** | Отличное, точное следование тексту | Лучшее в целом |
| **VRAM** | ~12 ГБ пик | ~15 ГБ пик |
| **Лучше для** | Быстрых итераций, текстовых промптов | Финального качества, I2V |

HunyuanVideo идеален, когда нужна **быстрая генерация** или хотите **поэкспериментировать с промптами** перед длительным рендером в Wan.

**Запускайте ячейки 1-6 по порядку**, затем откройте ссылку на туннель.

In [ ]:
#@title 1. Проверка GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} ГБ")

In [ ]:
#@title 2. Установка ComfyUI + Кастомные ноды
%cd /content
!git clone https://github.com/comfyanonymous/ComfyUI.git 2>/dev/null || echo "Уже склонировано"
%cd /content/ComfyUI
!pip install -r requirements.txt -q

# Кастомные ноды
%cd /content/ComfyUI/custom_nodes
!git clone https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git 2>/dev/null || echo "Уже склонировано"
!git clone https://github.com/kijai/ComfyUI-KJNodes.git 2>/dev/null || echo "Уже склонировано"

!pip install -r ComfyUI-VideoHelperSuite/requirements.txt -q
!pip install -r ComfyUI-KJNodes/requirements.txt -q

print("\nУстановлено!")

In [ ]:
#@title 3. Скачивание моделей HunyuanVideo (~15 ГБ)
import os

M = "/content/ComfyUI/models"
os.makedirs(f"{M}/diffusion_models", exist_ok=True)
os.makedirs(f"{M}/text_encoders", exist_ok=True)
os.makedirs(f"{M}/vae", exist_ok=True)
os.makedirs(f"{M}/clip", exist_ok=True)

files = {
    # HunyuanVideo transformer FP8 (~7.76 ГБ)
    f"{M}/diffusion_models/hunyuan_video_v2_replace_fp8_e4m3fn.safetensors":
        "https://huggingface.co/Comfy-Org/HunyuanVideo_repackaged/resolve/main/split_files/diffusion_models/hunyuan_video_v2_replace_fp8_e4m3fn.safetensors",
    # LLM текстовый энкодер FP8 (~4.5 ГБ)
    f"{M}/text_encoders/llava_llama3_fp8_scaled.safetensors":
        "https://huggingface.co/Comfy-Org/HunyuanVideo_repackaged/resolve/main/split_files/text_encoders/llava_llama3_fp8_scaled.safetensors",
    # CLIP текстовый энкодер (~246 МБ)
    f"{M}/clip/clip_l.safetensors":
        "https://huggingface.co/Comfy-Org/HunyuanVideo_repackaged/resolve/main/split_files/text_encoders/clip_l.safetensors",
    # VAE (~826 МБ)
    f"{M}/vae/hunyuan_video_vae_bf16.safetensors":
        "https://huggingface.co/Comfy-Org/HunyuanVideo_repackaged/resolve/main/split_files/vae/hunyuan_video_vae_bf16.safetensors",
}

for dest, url in files.items():
    if os.path.exists(dest):
        print(f"Уже существует: {os.path.basename(dest)}")
    else:
        print(f"\nСкачивание {os.path.basename(dest)}...")
        !wget -q --show-progress -O "{dest}" "{url}"

print("\nВсе модели готовы!")
!du -sh {M}/diffusion_models/* {M}/text_encoders/* {M}/clip/* {M}/vae/*

In [ ]:
#@title 4. Скачивание воркфлоу
import os

GIST = "ea1ab4a53e1be1b8401e5031bcdae4f3"
RAW = f"https://gist.githubusercontent.com/russiankendricklamar/{GIST}/raw"
WF_DIR = "/content/ComfyUI/user/default/workflows"
os.makedirs(WF_DIR, exist_ok=True)

!wget -q -O "{WF_DIR}/video_hunyuan_t2v.json" "{RAW}/video_hunyuan_t2v.json"
print("Воркфлоу скачан!")

In [ ]:
#@title 5. Загрузка медиа (для будущей поддержки I2V)
from google.colab import files
import os

INPUT_DIR = "/content/ComfyUI/input"
os.makedirs(INPUT_DIR, exist_ok=True)

uploaded = files.upload()
for name, data in uploaded.items():
    path = os.path.join(INPUT_DIR, name)
    with open(path, "wb") as f:
        f.write(data)
    print(f"Сохранено: {path}")

print(f"\nФайлы в input/: {os.listdir(INPUT_DIR)}")

In [ ]:
#@title 6. Запуск ComfyUI
import subprocess, time, re

# Установка cloudflared
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

# Запуск ComfyUI
comfy = subprocess.Popen(
    ["python", "main.py", "--listen", "0.0.0.0", "--port", "8188"],
    cwd="/content/ComfyUI",
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)
print("Запуск ComfyUI... (30 сек)")
time.sleep(30)

# Cloudflare туннель
tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8188"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)
time.sleep(5)

url = None
for _ in range(20):
    line = tunnel.stdout.readline().decode()
    match = re.search(r"https://[\w-]+\.trycloudflare\.com", line)
    if match:
        url = match.group(0)
        break

if url:
    print(f"\n{'='*60}")
    print(f"ComfyUI готов! Откройте: {url}")
    print(f"{'='*60}")
    print("\nЗагрузите воркфлоу: Меню -> Load -> video_hunyuan_t2v")
else:
    print("Ссылка на туннель не найдена. ComfyUI работает на порту 8188.")

In [ ]:
#@title 7. Сохранение на Google Drive
from google.colab import drive
import shutil, glob, os

drive.mount('/content/drive')

src = "/content/ComfyUI/output"
dst = "/content/drive/MyDrive/ComfyUI_Output/hunyuan"
os.makedirs(dst, exist_ok=True)

videos = glob.glob(f"{src}/*.mp4") + glob.glob(f"{src}/*.png")
if not videos:
    print("Результатов пока нет. Сначала сгенерируйте видео!")
else:
    for v in videos:
        shutil.copy2(v, dst)
        print(f"Скопировано: {os.path.basename(v)}")
    print(f"\n{len(videos)} файлов сохранено в {dst}")

---
## Руководство по использованию

### video_hunyuan_t2v — Текст-в-Видео
1. Откройте ссылку на туннель
2. Загрузите воркфлоу **video_hunyuan_t2v**
3. Отредактируйте текстовый промпт
4. Нажмите **Queue Prompt**

### Пресеты разрешений (16:9)
| Размер | Кадры | Длительность | VRAM |
|--------|-------|--------------|------|
| 848x480 | 61 | ~4 сек | ~12 ГБ |
| 848x480 | 97 | ~6 сек | ~14 ГБ |
| 640x368 | 129 | ~8 сек | ~13 ГБ |

### Советы
- HunyuanVideo очень точно следует текстовым инструкциям
- Используйте детальные, конкретные промпты для лучших результатов
- Держите количество кадров ниже 129 на T4, чтобы избежать OOM
- Выход — 16fps (против 24fps у Wan)

### Стиль промптов
HunyuanVideo лучше всего работает с **описательными промптами на естественном языке**:
> "A golden retriever running through a sunlit meadow, wildflowers swaying in the breeze, cinematic camera tracking shot, warm afternoon light"